### Chains

1. Analysis of downloads in 3 segments of model chains

In [2]:
import csv

chains_file = "../../data/model_chains/model_chains_0625.txt"
props_file = "../../data/model_props/model_props_0625.csv"

downloads = {}
with open(props_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        val = row.get("Download", "").strip()
        try:
            downloads[model_id] = int(val)
        except Exception:
            downloads[model_id] = 0

front_models = []
middle_models = []
back_models = []
chain_num = 0

with open(chains_file, 'r', encoding='utf-8') as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        parts = [p.strip() for p in raw.split("->") if p.strip()]
        if not parts:
            continue

        chain_num += 1
        n = len(parts)

        if n == 1:
            front_models.extend(parts)
        elif n == 2:
            front_models.append(parts[0])
            back_models.append(parts[1])
            middle_models.append((parts[0], parts[1]))
        else:
            third = n // 3
            front_part = parts[:third]
            middle_part = parts[third:2*third]
            back_part = parts[2*third:]
            front_models.extend(front_part)
            middle_models.extend(middle_part)
            back_models.extend(back_part)

def compute_stats(models_list):
    total = 0
    count = 0
    for m in models_list:
        if isinstance(m, tuple):
            val = (downloads.get(m[0],0) + downloads.get(m[1],0)) / 2
            total += val
            count += 1
        else:
            total += downloads.get(m,0)
            count += 1
    avg = total / count if count else 0
    return total, avg

front_total, front_avg = compute_stats(front_models)
middle_total, middle_avg = compute_stats(middle_models)
back_total, back_avg = compute_stats(back_models)

unique_front_total, _ = compute_stats(set(front_models))
unique_middle_total, _ = compute_stats(set(middle_models))
unique_back_total, _ = compute_stats(set(back_models))

print(f"Chains count: {chain_num}")
print(f"models in front segment: {len(front_models)}")
print(f"models in middle segment: {len(middle_models)}")
print(f"models in back segment: {len(back_models)}\n")

print(f"front segment downloads: {front_total}, avg: {front_avg:.2f}")
print(f"middle segment downloads: {middle_total}, ave: {middle_avg:.2f}")
print(f"back segment downloads: {back_total}, avg: {back_avg:.2f}")

Chains count: 1008871
models in front segment: 2023787
models in middle segment: 2023787
models in back segment: 2571391

front segment downloads: 949929505789, avg: 469382.16
middle segment downloads: 512346250670.5, ave: 253162.14
back segment downloads: 2645165865, avg: 1028.69


2. Analysis of dependency in 3 segments of model chains

In [1]:
import csv
from collections import Counter

chains_file = "../../data/model_chains/model_chains_0625.txt"
relations_file = "../../data/model_relations/batch_all_0625.csv"

model_type_map = {} 
with open(relations_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        mtype = row.get("Type", "").strip().lower()
        if mtype == "adapter":
            mtype = "finetune"
        elif mtype not in ("finetune", "quantized", "merge"):
            mtype = "none"
        if model_id:
            model_type_map[model_id] = mtype

front_models = []
middle_models = []
back_models = []
chain_count = 0

with open(chains_file, 'r', encoding='utf-8') as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        parts = [p.strip() for p in raw.split("->") if p.strip()]
        if not parts:
            continue
        chain_count += 1
        n = len(parts)

        if n == 1:
            front_models.append(parts[0])
        elif n == 2:
            front_models.append(parts[0])
            back_models.append(parts[1])
            middle_models.append((parts[0], parts[1]))  
        else:
            third = n // 3
            front_models.extend(parts[:third])
            middle_models.extend(parts[third:2*third])
            back_models.extend(parts[2*third:])

def compute_type_distribution(models_list):
    counter = Counter()
    for m in models_list:
        if isinstance(m, tuple):
            t1 = model_type_map.get(m[0], "none")
            t2 = model_type_map.get(m[1], "none")
            if t1 == t2:
                counter[t1] += 1
            else:
                counter[t1] += 0.5
                counter[t2] += 0.5
        else:
            counter[model_type_map.get(m, "none")] += 1
    total = sum(counter.values())
    return counter, total

for seg_name, seg_models in zip(["Front", "Middle", "Back"],
                                [front_models, middle_models, back_models]):
    counter, total = compute_type_distribution(seg_models)
    print(f"\n=== {seg_name} Segment ===")
    for t, c in counter.items():
        print(f"  {t}: {c:.1f} ({c/total*100:.2f}%)")


=== Front Segment ===
  none: 1008871.0 (49.85%)
  finetune: 157909.0 (7.80%)
  merge: 856109.0 (42.30%)
  quantized: 898.0 (0.04%)

=== Middle Segment ===
  none: 197509.0 (9.76%)
  finetune: 473763.5 (23.41%)
  quantized: 37876.0 (1.87%)
  merge: 1314638.5 (64.96%)

=== Back Segment ===
  finetune: 538735.0 (20.95%)
  quantized: 553342.0 (21.52%)
  merge: 1479314.0 (57.53%)


3. Analysis of update in 3 segments of model chains

In [4]:

chains_file = "../../data/model_chains/model_chains_0625.txt"
update_file = "../../data/implicit_update_models.txt"

with open(update_file, 'r', encoding='utf-8') as f:
    update_models = set(line.strip() for line in f if line.strip())

seg_models = {"front": [], "middle": [], "back": []}
seg_update_count = {"front": 0, "middle": 0, "back": 0}
chain_num = 0

with open(chains_file, 'r', encoding='utf-8') as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        parts = [p.strip() for p in raw.split("->") if p.strip()]
        if not parts:
            continue

        chain_num += 1
        n = len(parts)

        if n == 2:
            front_part = [parts[0]]
            back_part = [parts[1]]

            seg_models["front"].extend(front_part)
            seg_models["back"].extend(back_part)
            seg_update_count["front"] += sum(1 for m in front_part if m in update_models)
            seg_update_count["back"] += sum(1 for m in back_part if m in update_models)

            continue

        third = n // 3
        remainder = n % 3

        front_end = third + (1 if remainder > 0 else 0)
        middle_end = front_end + third + (1 if remainder > 1 else 0)

        front_part = parts[:front_end]
        middle_part = parts[front_end:middle_end]
        back_part = parts[middle_end:]

        for name, seg in zip(["front", "middle", "back"], [front_part, middle_part, back_part]):
            seg_models[name].extend(seg)
            seg_update_count[name] += sum(1 for m in seg if m in update_models)


if seg_models["middle"] == []:
    seg_models["middle"] = seg_models["front"] + seg_models["back"]
    seg_update_count["middle"] = (seg_update_count["front"] + seg_update_count["back"]) / 2


for seg in ["front", "middle", "back"]:
    total = len(seg_models[seg])
    upd = seg_update_count[seg]
    print(f"{seg} update model occurance: {upd}")


front update model occurance: 797283
middle update model occurance: 218326
back update model occurance: 80447


4. Analysis of addition in 3 segments of model chains

In [5]:
import csv

chains_file = "../../data/model_chains/model_chains_0625.txt"
add_file = "../../data/added_model_0625_from_0312.csv"

add_models = set()
with open(add_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        if model_id:
            add_models.add(model_id)

seg_models = {"front": [], "middle": [], "back": []}
seg_add_count = {"front": 0, "middle": 0, "back": 0}
chain_num = 0

with open(chains_file, 'r', encoding='utf-8') as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        parts = [p.strip() for p in raw.split("->") if p.strip()]
        if not parts:
            continue

        chain_num += 1
        n = len(parts)

        if n == 2:
            front_part = [parts[0]]
            back_part = [parts[1]]

            seg_models["front"].extend(front_part)
            seg_models["back"].extend(back_part)
            seg_add_count["front"] += sum(1 for m in front_part if m in add_models)
            seg_add_count["back"] += sum(1 for m in back_part if m in add_models)

            continue

        third = n // 3
        remainder = n % 3

        front_end = third + (1 if remainder > 0 else 0)
        middle_end = front_end + third + (1 if remainder > 1 else 0)

        front_part = parts[:front_end]
        middle_part = parts[front_end:middle_end]
        back_part = parts[middle_end:]

        for name, seg in zip(["front", "middle", "back"], [front_part, middle_part, back_part]):
            seg_models[name].extend(seg)
            seg_add_count[name] += sum(1 for m in seg if m in add_models)


if seg_models["middle"] == []: 
    seg_models["middle"] = seg_models["front"] + seg_models["back"]
    seg_add_count["middle"] = (seg_add_count["front"] + seg_add_count["back"]) / 2


for seg in ["front", "middle", "back"]:
    total = len(seg_models[seg])
    addc = seg_add_count[seg]
    print(f"{seg} add model occurance: {addc}")



front add model occurance: 32406
middle add model occurance: 95607
back add model occurance: 345799


5. Analysis of deletion in 3 segments of model chains

In [6]:
import csv

chains_file = "../../data/model_chains/model_chains_0312.txt"
delete_file = "../../data/deleted_model_0625_from_0312.csv"

delete_models = set()
with open(delete_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:

        model_id = row.get("Model ID", "").strip()
        if model_id:
            delete_models.add(model_id)

seg_models = {"front": [], "middle": [], "back": []}
seg_delete_count = {"front": 0, "middle": 0, "back": 0}
chain_num = 0

with open(chains_file, 'r', encoding='utf-8') as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        parts = [p.strip() for p in raw.split("->") if p.strip()]
        if not parts:
            continue

        chain_num += 1
        n = len(parts)

        if n == 2:
            front_part = [parts[0]]
            back_part = [parts[1]]

            seg_models["front"].extend(front_part)
            seg_models["back"].extend(back_part)
            seg_delete_count["front"] += sum(1 for m in front_part if m in delete_models)
            seg_delete_count["back"] += sum(1 for m in back_part if m in delete_models)

            continue

        third = n // 3
        remainder = n % 3

        front_end = third + (1 if remainder > 0 else 0)
        middle_end = front_end + third + (1 if remainder > 1 else 0)

        front_part = parts[:front_end]
        middle_part = parts[front_end:middle_end]
        back_part = parts[middle_end:]

        for name, seg in zip(["front", "middle", "back"], [front_part, middle_part, back_part]):
            seg_models[name].extend(seg)
            seg_delete_count[name] += sum(1 for m in seg if m in delete_models)

if seg_models["middle"] == []:  
    seg_models["middle"] = seg_models["front"] + seg_models["back"]
    seg_delete_count["middle"] = (seg_delete_count["front"] + seg_delete_count["back"]) / 2

for seg in ["front", "middle", "back"]:
    total = len(seg_models[seg])
    d = seg_delete_count[seg]
    print(f"{seg} delete model occurance: {d}")


front delete model occurance: 5562
middle delete model occurance: 3860
back delete model occurance: 23432


### Clusters

1. Analysis of downloads in 3 layers of model clusters

In [17]:
import csv
from collections import defaultdict

chains_file = "../../data/model_chains/model_chains_0312.txt"
props_file = "../../data/model_props/model_props_0625.csv"

downloads = {}
with open(props_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        val = row.get("Download", "").strip()
        try:
            downloads[model_id] = int(val)
        except:
            downloads[model_id] = 0

clusters = defaultdict(lambda: [[], [], []]) 
chain_count = 0

with open(chains_file, 'r', encoding='utf-8') as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        parts = [p.strip() for p in raw.split("->") if p.strip()]
        if not parts:
            continue

        chain_count += 1
        root_model = parts[0]
        n = len(parts)

        if n == 1:
            clusters[root_model][0].append(parts[0])
        elif n == 2:
            clusters[root_model][0].append(parts[0])
            clusters[root_model][2].append(parts[1])
         
        else:
            third = n // 3
            clusters[root_model][0].extend(parts[:third])
            clusters[root_model][1].extend(parts[third:2*third])
            clusters[root_model][2].extend(parts[2*third:])

def compute_stats_global(tiers_list):
    all_models = []
    for tier in tiers_list:
        all_models.extend(tier)
    unique_models = set(all_models)
    values = []
    for m in unique_models:
        if isinstance(m, tuple):
            val = sum(downloads.get(mid, 0) for mid in m) / len(m)
        else:
            val = downloads.get(m, 0)
        values.append(val)
    total = sum(values)
    avg = total / len(values) if values else 0
    return total, avg, len(unique_models)

front_total, front_avg, front_count = compute_stats_global([clusters[r][0] for r in clusters])
middle_total, middle_avg, middle_count = compute_stats_global([clusters[r][1] for r in clusters])
back_total, back_avg, back_count = compute_stats_global([clusters[r][2] for r in clusters])

print(f"cluster: {len(clusters)}\n")
print(f"top layer downloads: {front_total}, avg: {front_avg:.2f}, model num: {front_count}")
print(f"middle layer downloads: {middle_total}, avg: {middle_avg:.2f}, model num: {middle_count}")
print(f"bottom layer downloads:: {back_total}, avg: {back_avg:.2f}, model num: {back_count}")

cluster: 20256

top layer downloads: 1170873860, avg: 54542.97, model num: 21467
middle layer downloads: 73340976, avg: 6548.89, model num: 11199
bottom layer downloads:: 138287118, avg: 315.35, model num: 438518


In [18]:
import csv
from collections import Counter

relations_file = "../../data/model_relations/batch_all_0625.csv"
model_type_map = {} 
with open(relations_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        mtype = row.get("Type", "").strip().lower()
        if mtype == "adapter":
            mtype = "finetune"
        elif mtype not in ("finetune", "quantized", "merge"):
            mtype = "none"
        if model_id:
            model_type_map[model_id] = mtype

front_models = []
middle_models = []
back_models = []

for root_model, tiers in clusters.items():
    front_models.extend(tiers[0])
    middle_models.extend(tiers[1])
    back_models.extend(tiers[2])

def compute_type_distribution(models_list, dedup=True):

    counter = Counter()
    seen = set()
    
    for m in models_list:
        if dedup:
            if isinstance(m, tuple):
                key = tuple(sorted(m))
            else:
                key = m
            if key in seen:
                continue
            seen.add(key)

        if isinstance(m, tuple):
            t1 = model_type_map.get(m[0], "none")
            t2 = model_type_map.get(m[1], "none")
            if t1 == t2:
                counter[t1] += 1
            else:
                counter[t1] += 0.5
                counter[t2] += 0.5
        else:
            counter[model_type_map.get(m, "none")] += 1

    total = sum(counter.values())
    return counter, total

for seg_name, seg_models in zip(["Front", "Middle", "Back"],
                                [front_models, middle_models, back_models]):
    counter, total = compute_type_distribution(seg_models, dedup=True)
    print(f"\n=== {seg_name} Layer ===")
    for t, c in counter.items():
        print(f"  {t}: {c:.1f} ({c/total*100:.2f}%)")


=== Front Layer ===
  none: 20199.0 (94.09%)
  finetune: 463.0 (2.16%)
  merge: 781.0 (3.64%)
  quantized: 24.0 (0.11%)

=== Middle Layer ===
  finetune: 6210.0 (55.45%)
  none: 322.0 (2.88%)
  merge: 4365.0 (38.98%)
  quantized: 302.0 (2.70%)

=== Back Layer ===
  finetune: 338213.0 (77.13%)
  none: 19290.0 (4.40%)
  quantized: 72043.0 (16.43%)
  merge: 8972.0 (2.05%)


In [9]:
import csv

add_file = "../../data/added_model_0625_from_0312.csv"
add_models = set()
with open(add_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        if model_id:
            add_models.add(model_id)

tier_names = ["front", "middle", "back"]
tier_models = {name: [] for name in tier_names}
tier_add_count = {name: 0 for name in tier_names}

for root_model, tiers in clusters.items():
    for i, name in enumerate(tier_names):
        tier_models[name].extend(tiers[i])
        tier_add_count[name] += sum(1 for m in tiers[i] if m in add_models)

unique_models = {k: set(v) for k, v in tier_models.items()}
unique_add_count = {k: sum(1 for m in v if m in add_models) for k, v in unique_models.items()}

for name in tier_names:
    total = len(unique_models[name])
    addc = unique_add_count[name]
    pct = (addc / total * 100) if total else 0
    print(f"{name} cluster add model num: {addc} / {total} ({pct:.2f}%)")

front cluster add model num: 3472 / 26705 (13.00%)
middle cluster add model num: 2645 / 14193 (18.64%)
back cluster add model num: 102271 / 521907 (19.60%)


In [16]:
import csv

delete_file = "../../data/deleted_model_0625_from_0312.csv"
delete_models = set()
with open(delete_file, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model_id = row.get("Model ID", "").strip()
        if model_id:
            delete_models.add(model_id)

tier_names = ["front", "middle", "back"]
tier_models = {name: [] for name in tier_names}
tier_delete_count = {name: 0 for name in tier_names}

for root_model, tiers in clusters.items():
    for i, name in enumerate(tier_names):
        tier_models[name].extend(tiers[i])
        tier_delete_count[name] += sum(1 for m in tiers[i] if m in delete_models)

unique_models = {k: set(v) for k, v in tier_models.items()}
unique_delete_count = {k: sum(1 for m in v if m in delete_models) for k, v in unique_models.items()}

for name in tier_names:
    total = len(unique_models[name])
    delc = unique_delete_count[name]
    print(f"{name} cluster delete model {delc}")

front cluster delete model 403
middle cluster delete model 318
back cluster delete model 18731


In [10]:
update_file = "../../data/implicit_update_models.txt"
update_models = set()
with open(update_file, 'r', encoding='utf-8') as f:
    for line in f:
        model_id = line.strip()
        if model_id:
            update_models.add(model_id)

tier_names = ["front", "middle", "back"]
tier_models = {name: [] for name in tier_names}
tier_update_count = {name: 0 for name in tier_names}

for root_model, tiers in clusters.items():
    for i, name in enumerate(tier_names):
        tier_models[name].extend(tiers[i])
        tier_update_count[name] += sum(1 for m in tiers[i] if m in update_models)

unique_models = {k: set(v) for k, v in tier_models.items()}
unique_update_count = {k: sum(1 for m in v if m in update_models) for k, v in unique_models.items()}


for name in tier_names:
    total = len(unique_models[name])
    upd = unique_update_count[name]
    print(f"{name} cluster update model: {upd} / {total} ({upd/total*100:.2f}%)")

front cluster update model: 8900 / 26705 (33.33%)
middle cluster update model: 3111 / 14193 (21.92%)
back cluster update model: 34121 / 521907 (6.54%)
